In [1]:
import torch
print('PyTorch version:', torch.__version__)
from torch import optim
from torch.utils.data import DataLoader
import numpy as np

import torch
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as transforms

from torchvision.transforms import ToTensor

#Use in other files
# import os
# from torch import nn
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader
# from torchvision import datasets
# import torchvision.transforms as transforms
# from torchvision.transforms import ToTensor


#Dont know if i need until graphing and such
# import matplotlib.pyplot as plt
# from torchinfo import summary
# from torchvision.utils import make_grid
# from torchvision.io import decode_image


# from tqdm import tqdm
# import torchvision
# print('Torchvision version:', torchvision.__version__)


PyTorch version: 2.5.1+cu118


Load my other files

In [2]:
from ddpm_config import *
from dataloading import MCImageDataset
from diffusion_module import diffusion
from Unet_import import UNet
from sparsity import create_sparsity_mask, get_sparsity_mask #induece sparsity is for sampling

In [3]:

#initalize objects for training
diffusion_module = diffusion(startVariance=0, maxVariance=diffusion_parameters['max_variance'], spacing=diffusion_parameters['schedule_spacing'], diffusionSteps=diffusion_parameters['num_timesteps'], device=device)
model = UNet(t_dim=UNet_params['t_dim'], device=device)
optimizer = optim.AdamW(model.parameters(), lr = 0.001)

Training Cycle

In [4]:

#load datasets
training_dataset = MCImageDataset(images_path)
testing_dataset = MCImageDataset(images_path)

training_dataloader = DataLoader(training_dataset, batch_size=training_params['batch_size'], shuffle=True, drop_last=True)
test_dataloader = DataLoader(testing_dataset, batch_size=training_params['batch_size'], shuffle=False, drop_last=True)

#misc.
num_epochs = training_params["num_epochs"]
epoch_losses = []


#Training loop over all training epochs
for epoch in range(num_epochs):

    #losses is loss for a given batch
    losses = []

    #now go through evert image in the dataset
    for data in training_dataloader:

        model.train() 

        data = data.to(device) #Shape: B, C, H, W

        #generate stuff for the sparsity bits formulation of the diffusion model
        sparsity_mask = create_sparsity_mask(data) #Shape: B, 1, H, W
        data_and_mask = torch.cat([data, sparsity_mask], dim=1) #Shape: B, C+1, H, W
        
        #Create a time vector of shape (B,) will be extended in UNET in time embedding to (B, 2)
        t = torch.randint(low=0, high=diffusion_parameters['num_timesteps'], size=(data_and_mask.shape[0],)).to(device)
        
        #create noise epsilon of shape data_and_mask: B, C+1, H, W
        noise = torch.randn_like(data_and_mask) 
        
        #diffused noised data+mask at timestep t
        x_t = diffusion_module.forward(input=data_and_mask, noise=noise, timestep=t) 

        #get model prediction
        prediction = model(x_t, t)
        predicted_sparsity_mask, predicted_feature_map = get_sparsity_mask(prediction)

        #claculate losses depending on preferred prediction method specfiied in config
        if training_params['prediction_type'] == 'image':
            feature_map_loss = training_params['fmap_lossfunc'](data, predicted_feature_map)

        elif training_params['prediction_type'] == 'noise':
            feature_map_loss = training_params['fmap_lossfunc'](noise, predicted_feature_map)
 

        sbit_loss = training_params['sbit_lossfunc'](sparsity_mask, predicted_sparsity_mask)
        batch_loss = sbit_loss+feature_map_loss

        #Reset optimizer for next epoch
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()


        losses.append(batch_loss.item())
        
    training_per_epoch_loss = np.array(losses).mean()
    epoch_losses.append(training_per_epoch_loss)

#loss per epoch (mean of all batches losses). Into a tensor for saving
epoch_losses = torch.tensor(epoch_losses, dtype=torch.float32)

C:\Users\roboc\AppData\Local\Temp\ipykernel_11452\2217955943.py:62: RuntimeWarning: Mean of empty slice.
  training_per_epoch_loss = np.array(losses).mean()
c:\Users\roboc\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Save Model for Future Training and or Sampling


In [5]:
model_name = f"ddpm_{str(diffusion_parameters['schedule_spacing'])}_scheduling_{training_params['prediction_type']}_prediction"
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'overall_losses': epoch_losses,
    'scheduler_state_dict': diffusion_module.state_dict(),
    }, model_name) 
